In [ ]:
%pip install pandas>=2.2 networkx decorator tqdm six scipy
%pip install littleballoffur --no-deps

In [ ]:
pip install torch

In [ ]:
pip install torch_geometric

In [ ]:
pip install networkit

## Greedy Graph Coreset Selection Algorithm

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from littleballoffur import RandomWalkSampler, RandomNodeSampler, DegreeBasedSampler, PageRankBasedSampler, ForestFireSampler, MetropolisHastingsRandomWalkSampler, SnowBallSampler
import random
import warnings

warnings.filterwarnings("ignore")


def OMP_sampling(L,NIt):

    n, _ = np.shape(L)
    b = np.zeros((n,1))
    x = []

    for i in range(NIt):

        IND = np.zeros((n,1))
        idx_list = random.sample(range(n), k=100)
        xa = abs(L[:,idx_list].T.dot(b))
        mx = min(xa)

        IND = (xa<=mx)

        Ind = [index for index,value in enumerate(IND) if value]
        Ind = Ind[random.randrange(len(Ind))]
        x.append(idx_list[Ind])

        b = b - L[:,[idx_list[Ind]]]
        L[:,[idx_list[Ind]]] = np.inf*np.ones((n,1))

    return x

## Utility Function

In [ ]:
def cluster_imbalance(labels, sampled_nodes, n_clusters):
    # expected fraction if sampling were perfectly proportional
    total        = len(labels)
    cluster_size = np.array([np.sum(labels == k) for k in range(n_clusters)])
    expected_frac = len(sampled_nodes) / total  # shape (n_clusters,)

    # actual fraction of each cluster that was sampled
    sampled_labels = labels[list(sampled_nodes)]
    sampled_size   = np.array([np.sum(sampled_labels == k) for k in range(n_clusters)])
    actual_frac    = sampled_size / cluster_size  # shape (n_clusters,)

    # imbalance: mean absolute deviation from expected
    imbalance = np.mean(np.abs(actual_frac - expected_frac))

    ## print(f"{'Mean imbalance error:':<40} {imbalance*100:.2f}%")

    return imbalance

## Visualization

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx
from torch_geometric.datasets import PolBlogs
from torch_geometric.datasets import Amazon


# --- load Cora ---
dataset = Planetoid(root='/tmp/Cora', name='Cora')
## dataset = Amazon(root='/tmp/Amazon', name='Photo')
data = dataset[0]

# --- convert to networkx ---
G = to_networkx(data, to_undirected=True)
labels = data.y.numpy()  # 7 classes in Cora

# --- keep largest connected component ---
lcc = max(nx.connected_components(G), key=len)
G = G.subgraph(lcc).copy()
labels = np.array([labels[n] for n in G.nodes()])

# --- remap node ids to 0..N-1 ---
G = nx.convert_node_labels_to_integers(G)

# --- layout (use spring since no geometric positions) ---
pos = nx.spring_layout(G)

# --- subsample ---
rng = np.random.default_rng()
sampled_nodes = set(rng.choice(list(G.nodes()), size=100, replace=False))
non_sampled   = [n for n in G.nodes() if n not in sampled_nodes]

# --- cluster imbalance ---
imbalance = cluster_imbalance(labels, sampled_nodes, n_clusters=7)

# --- edge styling ---
edge_colors = ['#1F77B4' if (u in sampled_nodes and v in sampled_nodes) else '#B4B2A9'
               for u, v in G.edges()]
edge_widths  = [1.2      if (u in sampled_nodes and v in sampled_nodes) else 0.3
               for u, v in G.edges()]

bg_colors = [cm.tab10(labels[n]) for n in non_sampled]

fig, ax = plt.subplots(figsize=(6, 6))

nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=non_sampled,
                       node_color=bg_colors, node_size=10,
                       alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=list(sampled_nodes),
                       node_color='#1F77B4', node_size=40,
                       alpha=1.0, ax=ax)

# ax.set_title(f"Cora  ·  n={G.number_of_nodes()}, m={G.number_of_edges()}  ·  "
#              f"subsample k={len(sampled_nodes)}  ·  imbalance={imbalance*100:.1f}%")
ax.axis('off')
plt.tight_layout()
plt.show()

## Cluster Balance Error

In [ ]:
iter = 10

imbalance_error_omp = 0
imbalance_error_RN = 0
imbalance_error_PR = 0
imbalance_error_DB = 0
imbalance_error_MH = 0
imbalance_error_uniform = 0

num_sampled = 100

for i in range(iter):
    print(i)
    rng = np.random.default_rng()
    sampled_nodes_uniform = set(rng.choice(list(G.nodes()), size=num_sampled, replace=False))

    A = nx.adjacency_matrix(G)
    n, _ = np.shape(A)
    L = nx.laplacian_matrix(G)
    LL = nx.laplacian_matrix(G)
    sampled_nodes_omp = OMP_sampling(L,num_sampled)

    model = RandomNodeSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_RN = np.asarray(subsampled_graph.nodes)

    model = PageRankBasedSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_PR = np.asarray(subsampled_graph.nodes)

    model = DegreeBasedSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_DB = np.asarray(subsampled_graph.nodes)

    model = MetropolisHastingsRandomWalkSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_MH = np.asarray(subsampled_graph.nodes)

    imbalance_error_omp = imbalance_error_omp + cluster_imbalance(labels, sampled_nodes_omp, n_clusters=7)/iter
    imbalance_error_RN = imbalance_error_RN + cluster_imbalance(labels, sampled_nodes_RN, n_clusters=7)/iter
    imbalance_error_PR = imbalance_error_PR + cluster_imbalance(labels, sampled_nodes_PR, n_clusters=7)/iter
    imbalance_error_DB = imbalance_error_DB + cluster_imbalance(labels, sampled_nodes_DB, n_clusters=7)/iter
    imbalance_error_MH = imbalance_error_MH + cluster_imbalance(labels, sampled_nodes_MH, n_clusters=7)/iter
    imbalance_error_uniform = imbalance_error_uniform + cluster_imbalance(labels, sampled_nodes_uniform, n_clusters=7)/iter


print(imbalance_error_omp)
print(imbalance_error_RN)
print(imbalance_error_PR)
print(imbalance_error_DB)
print(imbalance_error_MH)
print(imbalance_error_uniform)